In [ ]:
!pip install ultralytics fastapi uvicorn python-multipart pyngrok nest-asyncio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 kB 4.2 MB/s eta 0:00:00


In [ ]:
import io
import asyncio
import uvicorn
from fastapi import FastAPI, UploadFile, File
from PIL import Image
from ultralytics import YOLO
from pyngrok import ngrok

# Load YOLOv8 models into GPU memory
print("Loading YOLOv8 models...")
model_detect = YOLO("yolov8_best_smartdetection.pt")
model_classify = YOLO("yolov8_best.pt")

app = FastAPI(title="EcoPay 5G - Vision API")

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    # Read uploaded image stream into PIL Image
    image_bytes = await file.read()
    image = Image.open(io.BytesIO(image_bytes))

    # Stage 1: Waste object detection
    res_detect = model_detect(image)
    if len(res_detect[0].boxes) == 0:
        return {"status": "NO_WASTE_DETECTED", "is_waste": False}

    # Crop detected region using bounding box coordinates
    box = res_detect[0].boxes[0].xyxy[0].tolist()
    cropped_img = image.crop((box[0], box[1], box[2], box[3]))

    # Stage 2: Material classification on cropped region
    res_classify = model_classify(cropped_img)
    if len(res_classify[0].boxes) > 0:
        final_class = model_classify.names[int(res_classify[0].boxes[0].cls)]
        confidence = float(res_classify[0].boxes[0].conf)
        return {
            "status": "SUCCESS",
            "is_waste": True,
            "detected_class": final_class,
            "confidence": confidence
        }

    return {"status": "CLASSIFICATION_FAILED", "is_waste": True}

# Configure public Ngrok tunnel
ngrok.set_auth_token("3IxMRIA1QG1aVXI6vHmKg1FNEnC_7LC45YRi3iper4wm9kNrH")
public_url = ngrok.connect(8000)
print(f"API Endpoint URL for Streamlit configuration: {public_url.public_url}/predict")

# Launch Uvicorn server compatible with Colab event loop
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Chargement des modèles YOLOv8...
✅ URL DE L'API À COPIER DANS STREAMLIT : https://supermom-undiluted-pelt.ngrok-free.dev/predict


INFO:     Started server process [1511]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


0: 640x512 1 non_dechet, 817.0ms
Speed: 28.4ms preprocess, 817.0ms inference, 41.5ms postprocess per image at shape (1, 3, 640, 512)



/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


0: 640x256 1 plastic, 320.9ms
Speed: 4.4ms preprocess, 320.9ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 256)
INFO:     41.248.118.233:0 - "POST /predict HTTP/1.1" 200 OK

0: 640x512 1 non_dechet, 812.7ms
Speed: 9.8ms preprocess, 812.7ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 512)

0: 640x256 1 plastic, 282.1ms
Speed: 7.8ms preprocess, 282.1ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 256)
INFO:     41.248.118.233:0 - "POST /predict HTTP/1.1" 200 OK
